# dcgan-normal-init-002 — worked example 2: Re-init Conv/ConvT weights and count how many parameters were resampled

> Worked example from [Delta Drills](https://delta-drills.vercel.app). Atom: `dcgan-normal-init-002`.

**This is a worked example — read it, run it, follow the reasoning.** It is study material, not a graded drill (no completion beacon). When the steps feel obvious, move to the faded version, then the full drill.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)
import torch.nn as nn
import matplotlib.pyplot as plt

## Concept

Applying the DCGAN `N(0, 0.02)` init only touches Conv2d / ConvTranspose2d weight tensors. A useful audit is to count exactly how many scalar parameters got resampled, which equals the sum of `m.weight.numel()` over every conv-family submodule. This confirms the init reached the layers you expected and skipped Linear / BatchNorm.

## Worked solution

**Step 1 — reseed for determinism.** We draw fresh `N(0, 0.02)` values, so call `t.manual_seed(0)` before any init so the run is reproducible.

**Step 2 — accumulate as you walk.** Iterate `model.modules()` (every submodule). Keep a running `total` counter.

**Step 3 — branch on the conv family.** For each `m`, `if isinstance(m, (nn.Conv2d, nn.ConvTranspose2d))`: do the in-place `nn.init.normal_(m.weight, 0.0, 0.02)` AND add `m.weight.numel()` to `total`. We add the count only inside the branch so non-conv layers contribute nothing.

**Step 4 — why numel.** `m.weight.numel()` is the product of the weight shape (out, in, kH, kW), i.e. the number of scalar weights that were just overwritten. Summing across layers gives the total resampled count.

**Step 5 — verify against a hand computation.** A `Conv2d(8, 16, 3)` has 16*8*3*3 = 1152 weights; a `ConvTranspose2d(16, 4, 4)` has 16*4*4*4 = 1024. The Linear is skipped. Expected total = 2176.

In [ ]:
def reinit_and_count(model: nn.Module) -> int:
    t.manual_seed(0)
    total = 0
    for m in model.modules():
        if isinstance(m, (nn.Conv2d, nn.ConvTranspose2d)):
            nn.init.normal_(m.weight, 0.0, 0.02)
            total += m.weight.numel()
    return total

net = nn.Sequential(
    nn.Conv2d(8, 16, 3),
    nn.ConvTranspose2d(16, 4, 4),
    nn.Linear(4, 2),
)
n = reinit_and_count(net)
print('resampled conv params:', n)
print('conv0 weight std     :', round(net[0].weight.std().item(), 4))